# TabPFN classification (local)

Local version of the [Prior Labs TabPFN demo](https://colab.research.google.com/github/PriorLabs/TabPFN/blob/main/examples/notebooks/TabPFN_Demo_Local.ipynb) on two binary tasks:

| Dataset | File | Target | Rows |
|---|---|---|---|
| IBM Telco customer churn | `WA_Fn-UseC_-Telco-Customer-Churn.csv` | `Churn` (Yes/No) | 7,043 |
| Delivery acceptance | `hyper_ackt-dataset.csv` | `hyper_ack` (0/1) | 11,118 |

[TabPFN](https://github.com/PriorLabs/TabPFN) is a tabular foundation model: `fit` stores the training table; `predict` runs a forward pass. **Do not scale or one-hot encode.** Pass mixed pandas dtypes. Missing values are allowed.

First run downloads model weights. **TabPFN-3** (current Colab default) needs a free Prior Labs token from [ux.priorlabs.ai/account](https://ux.priorlabs.ai/account). Without a token this notebook uses **TabPFN-2** (Apache-style weights, ~1,024 training rows). **Thinking mode** (section 6) is API-only via `tabpfn-client`.

## 1. Setup

In [ ]:
# One-time: python -m venv --system-site-packages .venv && .venv/bin/pip install tabpfn tabpfn-client
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import importlib
import train_tabpfn

importlib.reload(train_tabpfn)
from train_tabpfn import (
    default_max_train_rows,
    fit_random_forest,
    load_part1,
    load_tabpfn_token,
    load_telco,
    make_tabpfn_classifier,
    maybe_subsample_train,
    pick_device,
    pick_version,
)

ROOT = Path(".")
load_tabpfn_token()  # reads TABPFN_TOKEN from .env; Jupyter does not load .env by itself
DEVICE = pick_device()
VERSION = pick_version()  # v3 if TABPFN_TOKEN is set, else v2
MAX_TRAIN = default_max_train_rows(VERSION, DEVICE)
print(f"torch={torch.__version__}  device={DEVICE}  version={VERSION}  max_train={MAX_TRAIN}  token={'yes' if load_tabpfn_token() else 'no'}")

## 2. Telco customer churn

Drop `customerID`. Coerce blank `TotalCharges` to NaN (TabPFN handles missing values). Keep service/contract columns as pandas `category` — no one-hot encoding.

In [ ]:
X_telco, y_telco = load_telco(ROOT / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(X_telco.shape, "churn rate", float(y_telco.mean()))
display(X_telco.head())
display(y_telco.value_counts().rename("count").to_frame())

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X_telco, y_telco, test_size=0.2, random_state=42, stratify=y_telco
)
Xtr, ytr = maybe_subsample_train(Xtr, ytr, MAX_TRAIN)

clf_telco = make_tabpfn_classifier(VERSION, DEVICE, n_estimators=None)
clf_telco.fit(Xtr, ytr)
proba_telco = clf_telco.predict_proba(Xte)[:, 1]
pred_telco = (proba_telco >= 0.5).astype(int)

print("ROC AUC", roc_auc_score(yte, proba_telco))
print("Accuracy", accuracy_score(yte, pred_telco))
print(classification_report(yte, pred_telco, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(yte, proba_telco, ax=axes[0], name="TabPFN")
axes[0].set_title("Telco churn ROC")
ConfusionMatrixDisplay.from_predictions(yte, pred_telco, ax=axes[1], colorbar=False)
axes[1].set_title("Telco churn confusion matrix")
fig.tight_layout()

## 3. Delivery `hyper_ack`

`hyper_ackt-dataset.csv` is a logistics table. Target `hyper_ack` is already 0/1. Drop `first_created_at` (almost unique timestamps). Convert `created_date` to day-of-year; keep `weekday` and `deliverey_category_id` as categories.

In [ ]:
X_part1, y_part1 = load_part1(ROOT / "hyper_ackt-dataset.csv")
print(X_part1.shape, "hyper_ack rate", float(y_part1.mean()))
display(X_part1.head())
display(y_part1.value_counts().rename("count").to_frame())

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X_part1, y_part1, test_size=0.2, random_state=42, stratify=y_part1
)
Xtr, ytr = maybe_subsample_train(Xtr, ytr, MAX_TRAIN)

clf_part1 = make_tabpfn_classifier(VERSION, DEVICE, n_estimators=None)
clf_part1.fit(Xtr, ytr)
proba_part1 = clf_part1.predict_proba(Xte)[:, 1]
pred_part1 = (proba_part1 >= 0.5).astype(int)

print("ROC AUC", roc_auc_score(yte, proba_part1))
print("Accuracy", accuracy_score(yte, pred_part1))
print(classification_report(yte, pred_part1, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(yte, proba_part1, ax=axes[0], name="TabPFN")
axes[0].set_title("hyper_ack ROC")
ConfusionMatrixDisplay.from_predictions(yte, pred_part1, ax=axes[1], colorbar=False)
axes[1].set_title("hyper_ack confusion matrix")
fig.tight_layout()

## 4. RandomForest baseline (optional)

Same splits, ordinal encoding for the forest only. TabPFN still gets the raw frame.

In [ ]:
def split(X, y):
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

for name, X, y in [
    ("telco", X_telco, y_telco),
    ("part1", X_part1, y_part1),
]:
    Xtr, Xte, ytr, yte = split(X, y)
    rf = fit_random_forest(Xtr, ytr, Xte, yte)
    print(name, "RF ROC-AUC", round(rf["roc_auc"], 4), "acc", round(rf["accuracy"], 4), "f1", round(rf["f1"], 4))

## CLI

```bash
source .venv/bin/activate
python train_tabpfn.py                 # TabPFN-2 if no token
python benchmark_tabpfn.py             # TabPFN vs trees / sklearn / TabTransformer
python benchmark_tabpfn.py --only-tab-transformer   # lucidrains TabTransformer + FTTransformer, merge into CSV
export TABPFN_TOKEN=...               # from https://ux.priorlabs.ai/account
python train_tabpfn.py --version v3
python train_tabpfn.py --version thinking   # TabPFN-3-Plus API thinking mode
python benchmark_tabpfn.py --include-thinking
```

## 5. Benchmark vs XGBoost / LightGBM / sklearn / TabTransformer

Prior Labs protocol: [docs.priorlabs.ai/benchmarking](https://docs.priorlabs.ai/benchmarking). Same 80/20 stratified split (`random_state=42`) for every model. Primary metric is ROC-AUC.

- **context**: all models see 1,024 train rows (TabPFN-2's recommended budget)
- **full**: trees, linear models, and TabTransformer / FTTransformer see the entire train split; TabPFN-2 stays at 1,024

TabTransformer and FTTransformer come from [lucidrains/tab-transformer-pytorch](https://github.com/lucidrains/tab-transformer-pytorch) (section 7). Run `python benchmark_tabpfn.py --only-tab-transformer` to refresh those rows without dropping TabPFN thinking scores.

In [ ]:
bench = pd.read_csv(ROOT / "results" / "benchmark.csv")
cols = ["dataset", "protocol", "train_n", "model", "roc_auc", "avg_precision", "accuracy", "f1", "fit_seconds"]
for dataset, g in bench.groupby("dataset", sort=False):
    print("\n====", dataset, "====")
    show = g[cols].sort_values(["protocol", "roc_auc"], ascending=[True, False])
    display(show.style.format({"roc_auc": "{:.3f}", "avg_precision": "{:.3f}", "accuracy": "{:.3f}", "f1": "{:.3f}", "fit_seconds": "{:.2f}"}).hide(axis="index"))

## 6. TabPFN thinking mode (new version, API)

This is a **different estimator** from local `tabpfn.TabPFNClassifier`. Import it from `tabpfn_client`. [Thinking mode](https://docs.priorlabs.ai/capabilities/thinking-mode) runs on hosted TabPFN-3-Plus: extra compute at `fit()`, then normal `predict` / `predict_proba`.

Jupyter does **not** load `.env`. Each cell below reads `TABPFN_TOKEN` from the project `.env` and calls `tabpfn_client.set_access_token(...)`. Re-run the cell; a kernel restart is not required. Quota is separate from local inference (default 20 thinking fits / month). Same 80/20 split as sections 2–3 (`random_state=42`). No scaling or one-hot encoding. Fit takes several minutes.

In [ ]:
from pathlib import Path
import os
from tabpfn_client import TabPFNClassifier, set_access_token

# Jupyter does not load .env. Register the token here (never printed).
_env_paths = [Path.cwd() / ".env", Path("/Users/shahriar/Desktop/Desktop/Work/MyStartUp/R&D/.env")]
_token = (os.environ.get("TABPFN_TOKEN") or "").strip()
_env_used = None
for _p in _env_paths:
    if not _p.is_file():
        continue
    for _line in _p.read_text().splitlines():
        _s = _line.strip()
        if _s.startswith("TABPFN_TOKEN="):
            _token = _s.split("=", 1)[1].strip().strip('"').strip("'")
            _env_used = _p
            break
    if _env_used is not None:
        break
if not _token:
    raise RuntimeError("TABPFN_TOKEN not found. Add TABPFN_TOKEN=... to the project .env, then re-run this cell.")
os.environ["TABPFN_TOKEN"] = _token
set_access_token(_token)
print("TabPFN token registered from", _env_used or "environment")

X_train, X_test, y_train, y_test = train_test_split(
    X_telco, y_telco, test_size=0.2, random_state=42, stratify=y_telco
)

clf = TabPFNClassifier(
    thinking_mode=True,
    thinking_effort="high",
    thinking_metric="accuracy",
)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)

print("Telco thinking  ROC AUC", roc_auc_score(y_test, probs[:, 1]))
print("Telco thinking  Accuracy", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, digits=4, zero_division=0))

In [ ]:
from pathlib import Path
import os
from tabpfn_client import TabPFNClassifier, set_access_token

_env_paths = [Path.cwd() / ".env", Path("/Users/shahriar/Desktop/Desktop/Work/MyStartUp/R&D/.env")]
_token = (os.environ.get("TABPFN_TOKEN") or "").strip()
_env_used = None
for _p in _env_paths:
    if not _p.is_file():
        continue
    for _line in _p.read_text().splitlines():
        _s = _line.strip()
        if _s.startswith("TABPFN_TOKEN="):
            _token = _s.split("=", 1)[1].strip().strip('"').strip("'")
            _env_used = _p
            break
    if _env_used is not None:
        break
if not _token:
    raise RuntimeError("TABPFN_TOKEN not found. Add TABPFN_TOKEN=... to the project .env, then re-run this cell.")
os.environ["TABPFN_TOKEN"] = _token
set_access_token(_token)
print("TabPFN token registered from", _env_used or "environment")

X_train, X_test, y_train, y_test = train_test_split(
    X_part1, y_part1, test_size=0.2, random_state=42, stratify=y_part1
)

clf = TabPFNClassifier(
    thinking_mode=True,
    thinking_effort="high",
    thinking_metric="accuracy",
)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)

print("hyper_ack thinking  ROC AUC", roc_auc_score(y_test, probs[:, 1]))
print("hyper_ack thinking  Accuracy", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, digits=4, zero_division=0))

## 7. TabTransformer and FTTransformer

Install: `pip install tab-transformer-pytorch`. Backbone: [lucidrains/tab-transformer-pytorch](https://github.com/lucidrains/tab-transformer-pytorch). Training follows the original papers plus:

- [Suzuki Kaggle FT-Transformer notebook](https://www.kaggle.com/code/masatakasuzuki/ft-transformer-transformer-for-tabular-data) / Gorishniy et al. 2021 — Feature Tokenizer (one token per column + CLS), `QuantileTransformer(output_distribution="normal")` on numerics, AdamW `lr=1e-4`, `wd=1e-5` with no decay on embeddings / LayerNorm / bias, patience 16, no LR schedule. Default size: `d_token=192`, 3 blocks, 8 heads.
- [Khoeini, FTTransformer](https://arashk.medium.com/fttransformer-transformer-architecture-for-tabular-datasets-d4bfe591d6fb) — each categorical and numeric feature is embedded; self-attention mixes **columns**, not rows.
- [Kolli, TabTransformer](https://aravindkolli.medium.com/mastering-tabular-data-with-tabtransformer-a-comprehensive-guide-119f6dbf5a79) — StandardScaler on numerics, dropout, Adam-family `lr=1e-3`, short LR warmup. That article's Wine-Quality code maps all features to sequence length 1; **we do not use that**. Huang et al. attend over categorical column embeddings, then concatenate scaled numerics into an MLP.

Same 80/20 `random_state=42` holdout as sections 2–3. CLI: `python benchmark_tabpfn.py --only-tab-transformer`.

In [ ]:
from tab_transformer_model import run_ft_transformer, run_tab_transformer

X_train, X_test, y_train, y_test = train_test_split(
    X_telco, y_telco, test_size=0.2, random_state=42, stratify=y_telco
)
for runner in (run_tab_transformer, run_ft_transformer):
    row = runner(X_train, y_train, X_test, y_test, device=DEVICE)
    print(
        "Telco",
        row["model"],
        "ROC-AUC",
        round(row["roc_auc"], 4),
        "acc",
        round(row["accuracy"], 4),
        "f1",
        round(row["f1"], 4),
        "fit",
        row["fit_seconds"],
        "s",
    )

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_part1, y_part1, test_size=0.2, random_state=42, stratify=y_part1
)
for runner in (run_tab_transformer, run_ft_transformer):
    row = runner(X_train, y_train, X_test, y_test, device=DEVICE)
    print(
        "hyper_ack",
        row["model"],
        "ROC-AUC",
        round(row["roc_auc"], 4),
        "acc",
        round(row["accuracy"], 4),
        "f1",
        round(row["f1"], 4),
        "fit",
        row["fit_seconds"],
        "s",
    )

In [ ]:
bench = pd.read_csv(ROOT / "results" / "benchmark.csv")
cols = ["dataset", "protocol", "train_n", "model", "roc_auc", "avg_precision", "accuracy", "f1", "fit_seconds"]
tt = bench[bench["model"].isin(["TabTransformer", "FTTransformer"])]
print("lucidrains transformers only")
display(
    tt[cols]
    .sort_values(["dataset", "protocol", "roc_auc"], ascending=[True, True, False])
    .style.format({"roc_auc": "{:.3f}", "avg_precision": "{:.3f}", "accuracy": "{:.3f}", "f1": "{:.3f}", "fit_seconds": "{:.2f}"})
    .hide(axis="index")
)

In [ ]:
from tabpfn import TabPFNClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = TabPFNClassifier()
# To use TabPFNv2:
# model = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
model.fit(X_train, y_train)

# Predict class labels
preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))

# Predict class probabilities
probs = model.predict_proba(X_test)
print("Log-Loss:", log_loss(y_test, probs))